In [30]:
import ollama

In [31]:
def generate_llm_financial_interpretation(summary):

    prompt = f"""
You are a senior equity research analyst.

Interpret the following Apple financial analysis.

def interpret_liabilities_ratio(ratio):

    if ratio < 50:
        return "The company maintains relatively low liabilities compared to assets."

    elif ratio < 90:
        return "The company carries substantial liabilities relative to assets, but liabilities remain below total assets."

    else:
        return "The company carries very high liabilities relative to assets."
        
Rules:
1. Do not make claims not directly supported by the metrics.
2. Do not compare with industry averages unless benchmark data is provided.
3. Do not say liabilities exceed assets unless ratio > 100%.
4. Use cautious analyst-style language.
5. Separate observations from conclusions.

Focus on:
- profitability
- growth
- efficiency
- financial health

Keep the answer concise but insightful.
Do not compare with industry average unless benchmark data is provided.

Financial summary:
{summary}
"""

    response = ollama.chat(
        model="mistral",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        options={
            "temperature": 0.2
        }
    )

    return response["message"]["content"]

In [32]:
import json

In [33]:
metrics_path = "D:/sudhendra/learning projects/GraphRAG/data/apple_2023_financial_metrics.json"

with open(metrics_path, "r") as f:
    metrics = json.load(f)

In [34]:
def get_value(metric_key, year):
    return metrics[metric_key]["values"][str(year)]


def percentage_change(current, previous):
    return ((current - previous) / previous) * 100


def margin(part, total):
    return (part / total) * 100


def safe_round(value):
    return round(value, 2)


def format_money(value):
    return f"${value:,.0f} million"


def format_percent(value):
    return f"{value:.2f}%"

In [35]:
def generate_financial_analysis(year=2023):
    
    previous_year = year - 1

    # Revenue
    revenue = get_value("total_net_sales", year)
    prev_revenue = get_value("total_net_sales", previous_year)

    revenue_growth = percentage_change(revenue, prev_revenue)

    # Net income
    net_income = get_value("net_income", year)
    prev_net_income = get_value("net_income", previous_year)

    net_income_growth = percentage_change(
        net_income,
        prev_net_income
    )

    # Margins
    gross_margin_value = get_value("gross_margin", year)
    operating_income = get_value("operating_income", year)

    gross_margin_pct = margin(
        gross_margin_value,
        revenue
    )

    operating_margin_pct = margin(
        operating_income,
        revenue
    )

    net_profit_margin_pct = margin(
        net_income,
        revenue
    )

    # Expenses
    rd = get_value("research_and_development", year)

    sga = get_value(
        "selling_general_and_administrative",
        year
    )

    total_opex = get_value(
        "total_operating_expenses",
        year
    )

    rd_pct = margin(rd, revenue)

    sga_pct = margin(sga, revenue)

    opex_pct = margin(total_opex, revenue)

    # Balance sheet
    assets = get_value("total_assets", year)

    liabilities = get_value(
        "total_liabilities",
        year
    )

    cash = get_value(
        "cash_and_cash_equivalents",
        year
    )

    liabilities_to_assets = margin(
        liabilities,
        assets
    )

    cash_to_assets = margin(
        cash,
        assets
    )

    analysis = {
        "year": year,

        "revenue": {
            "value": revenue,
            "growth_percent": safe_round(revenue_growth)
        },

        "net_income": {
            "value": net_income,
            "growth_percent": safe_round(net_income_growth)
        },

        "profitability": {
            "gross_margin_percent": safe_round(gross_margin_pct),
            "operating_margin_percent": safe_round(operating_margin_pct),
            "net_profit_margin_percent": safe_round(net_profit_margin_pct)
        },

        "expense_efficiency": {
            "rd_as_percent_of_sales": safe_round(rd_pct),
            "sga_as_percent_of_sales": safe_round(sga_pct),
            "operating_expenses_as_percent_of_sales": safe_round(opex_pct)
        },

        "balance_sheet": {
            "liabilities_to_assets_percent": safe_round(liabilities_to_assets),
            "cash_to_assets_percent": safe_round(cash_to_assets)
        }
    }

    return analysis

In [36]:
def detect_calculation_intent(question):
    q = question.lower()

    if "liabilities" in q and "assets" in q:
        return "liabilities_vs_assets"

    if "net profit margin" in q:
        return "net_profit_margin"

    if "gross margin" in q and "%" in q:
        return "gross_margin_percent"

    if "operating margin" in q:
        return "operating_margin"

    if "r&d" in q or "research and development" in q:
        if "%" in q or "percentage" in q:
            return "rd_as_percent_of_sales"
        return "research_and_development"

    if "revenue growth" in q or "sales growth" in q:
        return "revenue_growth"

    if "compare" in q and "net sales" in q:
        return "net_sales_comparison"

    if "liabilities to assets" in q:
        return "liabilities_to_assets"

    if "cash to assets" in q:
        return "cash_to_assets"

    return "rag_only"

In [37]:
def calculate_answer(question):
    intent = detect_calculation_intent(question)

    if intent == "liabilities_vs_assets":

        ratio = margin(
            get_value("total_liabilities", 2023),
            get_value("total_assets", 2023)
        )

        if ratio > 100:
            interpretation = (
            "Liabilities exceed total assets."
            )
        else:
            interpretation = (
            "Liabilities do not exceed total assets."
        )

        return {
            "intent": intent,
            "answer": interpretation,
            "ratio": ratio
        }

    if intent == "net_profit_margin":
        revenue = get_value("total_net_sales", 2023)
        net_income = get_value("net_income", 2023)
        result = margin(net_income, revenue)

        return {
            "intent": intent,
            "answer": f"Apple's net profit margin in 2023 was {format_percent(result)}.",
            "calculation": f"Net profit margin = Net income / Total net sales × 100 = {format_money(net_income)} / {format_money(revenue)} × 100 = {format_percent(result)}",
            "values_used": {
                "net_income_2023": net_income,
                "total_net_sales_2023": revenue
            }
        }

    if intent == "gross_margin_percent":
        revenue = get_value("total_net_sales", 2023)
        gross_margin = get_value("gross_margin", 2023)
        result = margin(gross_margin, revenue)

        return {
            "intent": intent,
            "answer": f"Apple's gross margin percentage in 2023 was {format_percent(result)}.",
            "calculation": f"Gross margin % = Gross margin / Total net sales × 100 = {format_money(gross_margin)} / {format_money(revenue)} × 100 = {format_percent(result)}",
            "values_used": {
                "gross_margin_2023": gross_margin,
                "total_net_sales_2023": revenue
            }
        }

    if intent == "operating_margin":
        revenue = get_value("total_net_sales", 2023)
        operating_income = get_value("operating_income", 2023)
        result = margin(operating_income, revenue)

        return {
            "intent": intent,
            "answer": f"Apple's operating margin in 2023 was {format_percent(result)}.",
            "calculation": f"Operating margin = Operating income / Total net sales × 100 = {format_money(operating_income)} / {format_money(revenue)} × 100 = {format_percent(result)}",
            "values_used": {
                "operating_income_2023": operating_income,
                "total_net_sales_2023": revenue
            }
        }

    if intent == "rd_as_percent_of_sales":
        revenue = get_value("total_net_sales", 2023)
        rd = get_value("research_and_development", 2023)
        result = margin(rd, revenue)

        return {
            "intent": intent,
            "answer": f"Apple's R&D expense as a percentage of sales in 2023 was {format_percent(result)}.",
            "calculation": f"R&D as % of sales = Research and development / Total net sales × 100 = {format_money(rd)} / {format_money(revenue)} × 100 = {format_percent(result)}",
            "values_used": {
                "research_and_development_2023": rd,
                "total_net_sales_2023": revenue
            }
        }

    if intent == "revenue_growth":
        revenue_2023 = get_value("total_net_sales", 2023)
        revenue_2022 = get_value("total_net_sales", 2022)

        change = revenue_2023 - revenue_2022
        pct_change = percentage_change(revenue_2023, revenue_2022)

        return {
            "intent": intent,
            "answer": f"Apple's total net sales decreased by {format_percent(abs(pct_change))} in 2023 compared with 2022.",
            "calculation": f"Revenue growth = (2023 net sales - 2022 net sales) / 2022 net sales × 100 = ({format_money(revenue_2023)} - {format_money(revenue_2022)}) / {format_money(revenue_2022)} × 100 = {format_percent(pct_change)}",
            "values_used": {
                "total_net_sales_2023": revenue_2023,
                "total_net_sales_2022": revenue_2022,
                "change": change
            }
        }

    if intent == "net_sales_comparison":
        revenue_2023 = get_value("total_net_sales", 2023)
        revenue_2022 = get_value("total_net_sales", 2022)

        change = revenue_2023 - revenue_2022
        pct_change = percentage_change(revenue_2023, revenue_2022)

        return {
            "intent": intent,
            "answer": f"Apple's total net sales decreased from {format_money(revenue_2022)} in 2022 to {format_money(revenue_2023)} in 2023.",
            "calculation": f"Difference = {format_money(revenue_2023)} - {format_money(revenue_2022)} = {format_money(change)}. Percentage change = {format_percent(pct_change)}.",
            "values_used": {
                "total_net_sales_2023": revenue_2023,
                "total_net_sales_2022": revenue_2022,
                "change": change
            }
        }

    if intent == "liabilities_to_assets":
        liabilities = get_value("total_liabilities", 2023)
        assets = get_value("total_assets", 2023)
        result = margin(liabilities, assets)

        return {
            "intent": intent,
            "answer": f"Apple's liabilities-to-assets ratio in 2023 was {format_percent(result)}.",
            "calculation": f"Liabilities to assets = Total liabilities / Total assets × 100 = {format_money(liabilities)} / {format_money(assets)} × 100 = {format_percent(result)}",
            "values_used": {
                "total_liabilities_2023": liabilities,
                "total_assets_2023": assets
            }
        }

    if intent == "cash_to_assets":
        cash = get_value("cash_and_cash_equivalents", 2023)
        assets = get_value("total_assets", 2023)
        result = margin(cash, assets)

        return {
            "intent": intent,
            "answer": f"Apple's cash-to-assets ratio in 2023 was {format_percent(result)}.",
            "calculation": f"Cash to assets = Cash and cash equivalents / Total assets × 100 = {format_money(cash)} / {format_money(assets)} × 100 = {format_percent(result)}",
            "values_used": {
                "cash_and_cash_equivalents_2023": cash,
                "total_assets_2023": assets
            }
        }

    return None

In [38]:
def answer_financial_question(question):

    # Step 1 — Try deterministic calculator
    calculation_result = calculate_answer(question)

    if calculation_result is not None:

        prompt = f"""
You are a financial analyst.

Use ONLY the provided calculation result.

Question:
{question}

Calculation Result:
{json.dumps(calculation_result, indent=4)}

Write a concise financial answer.

Rules:
1. Do not invent numbers.
2. Do not add unsupported claims.
3. Keep answer factual and concise.
"""

        response = ollama.chat(
            model="mistral",
            messages=[
                {
                    "role": "user",
                    "content": prompt
                }
            ],
            options={
                "temperature": 0.1
            }
        )

        return response["message"]["content"]

    # Step 2 — fallback for unsupported questions
    return "Question type not supported yet."

In [39]:
print(
    answer_financial_question(
        "What was Apple's net profit margin in 2023?"
    )
)

 Apple's net profit margin in 2023 was 25.31%. This calculation is based on the net income of $96,995 million and total net sales of $383,285 million for that year.


In [42]:
test_questions = [
    {
        "question": "What was Apple's net profit margin in 2023?",
        "expected_keywords": ["25.31", "net income", "total net sales"]
    },
    {
        "question": "What was Apple's operating margin in 2023?",
        "expected_keywords": ["29.82", "operating income"]
    },
    {
        "question": "Did Apple's liabilities exceed assets?",
        "expected_keywords": ["do not exceed", "below 100"]
    },
    {
        "question": "Compare Apple's 2023 and 2022 net sales.",
        "expected_keywords": ["383,285", "394,328", "decreased", "2.8"]
    },
    {
        "question": "Did Apple's liabilities exceed assets?",
        "expected_keywords": ["do not exceed", "82.37"]
    }
]

In [43]:
for test in test_questions:
    response = answer_financial_question(test["question"])

    print("QUESTION:", test["question"])
    print("RESPONSE:", response)

    for keyword in test["expected_keywords"]:
        print(keyword, "=>", keyword.lower() in response.lower())

    print("-" * 100)

QUESTION: What was Apple's net profit margin in 2023?
RESPONSE:  Apple's net profit margin in 2023 was 25.31%. This calculation is based on the net income of $96,995 million divided by total net sales of $383,285 million for that year.
25.31 => True
net income => True
total net sales => True
----------------------------------------------------------------------------------------------------
QUESTION: What was Apple's operating margin in 2023?
RESPONSE:  Apple's operating margin in 2023 was 29.82%. This figure is calculated as the operating income ($114,301 million) divided by total net sales ($383,285 million), then multiplied by 100.
29.82 => True
operating income => True
----------------------------------------------------------------------------------------------------
QUESTION: Did Apple's liabilities exceed assets?
RESPONSE:  Based on the provided calculation result, Apple's liabilities do not exceed total assets. The liabilities-to-assets ratio is 82.37%.
do not exceed => True
be

Testing after the whole pipeline is built and before we move towards the GRAPHRAG

In [1]:
import sys
sys.path.insert(0, '..')

from pathlib import Path
from pipeline import run_full_pipeline
from src.evaluator import run_evaluation



d:\sudhendra\learning projects\GraphRAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import importlib
import src.evaluator
importlib.reload(src.evaluator)

from src.evaluator import run_evaluation


In [7]:
apple_result = run_full_pipeline(
    company_name   = "apple",
    pdf_path       = "../data/Apple/Apple23 report.pdf",
    year           = 2023,
    rebuild_vectors = False,
)



  FULL PIPELINE: APPLE  |  FY2023

[Step 1/5] Detecting financial statement pages...
  Auto-detected income statement pages : [30, 31, 49, 57]
  Auto-detected balance sheet pages    : [33, 52]
  Combined target pages: [30, 31, 33, 49, 52, 57]

[Step 2/5] Loading/generating metric mapping config...
  Existing mapping config found: apple.json

[Step 3/5] Extracting metrics from PDF...

  Company : APPLE
  PDF     : Apple23 report.pdf
  Year    : 2023  |  Columns: [2023, 2022, 2021]
  Pages   : [30, 31, 33, 49, 52, 57]

[Step 1/4] Extracting financial lines from PDF...
  Opened PDF: Apple23 report.pdf (80 pages total)
  Extracted 95 candidate financial lines (139 rows skipped)

[Step 2/4] Assigning fiscal years to numeric columns...
  Year mapping complete: 95 rows with 3-year values

[Step 3/4] Saving raw extraction CSV for inspection...
  [Debug CSV saved] -> ..\data\Apple\apple_2023_raw_extraction.csv

[Step 4/4] Normalizing labels using mapping config...

Normalizing 95 rows for 'app

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3800.22it/s]


  Found existing collection 'apple_2023' (666 documents)
  Collection already populated. Skipping embedding step.
  (Pass rebuild=True to force re-embedding)

  Pipeline complete: APPLE FY2023
  Metrics ready    : ['total_revenue', 'gross_profit', 'research_and_development', 'selling_general_administrative', 'total_operating_expenses', 'operating_income', 'net_income', 'cash_and_equivalents', 'total_assets', 'total_liabilities']
  Vector store     : 666 documents in ChromaDB



In [8]:
apple_score = run_evaluation(
    eval_path       = "../data/eval/apple_eval.json",
    pipeline_result = apple_result,
    company         = "Apple",
)



EVALUATION REPORT  APPLE

[PASS] What was Apple's total revenue in 2023?
       Expected: 383285.0 | Found in response: [2023.0, 383285.0, 2023.0, 383285.0, 383285.0, 2023.0, 2023.0, 1.0, 31.0]

[PASS] What was Apple's gross margin in 2023?
       Expected: 44.13 | Found in response: [2023.0, 44.13, 2023.0, 169148.0, 2023.0, 383285.0, 100.0, 169148.0, 383285.0, 100.0, 44.13, 2023.0, 44.13, 44.13, 2.0, 26.0]

[PASS] What was Apple's net income in 2023?
       Expected: 96995.0 | Found in response: [2023.0, 96995.0, 2023.0, 96995.0, 96995.0, 2023.0, 25.31, 4.0, 31.0]

[PASS] What was Apple's operating margin in 2023?
       Expected: 29.82 | Found in response: [2023.0, 29.82, 2023.0, 114301.0, 2023.0, 383285.0, 100.0, 114301.0, 383285.0, 100.0, 29.82, 2023.0, 29.82, 29.82, 1.0, 52.0, 3.0, 31.0]

[PASS] What was Apple's net margin in 2023?
       Expected: 25.31 | Found in response: [2023.0, 25.31, 2023.0, 96995.0, 2023.0, 383285.0, 100.0, 96995.0, 383285.0, 100.0, 25.31, 2023.0, 25.31]


In [3]:
tesla_result = run_full_pipeline(
    company_name   = "tesla",
    pdf_path       = "../data/Tesla/tesla 22 10-k.pdf",
    year           = 2022,
    rebuild_vectors = False,
)



  FULL PIPELINE: TESLA  |  FY2022

[Step 1/5] Detecting financial statement pages...
  Auto-detected income statement pages : [49, 80]
  Auto-detected balance sheet pages    : [48]
  Combined target pages: [48, 49, 80]

[Step 2/5] Loading/generating metric mapping config...
  Existing mapping config found: tesla.json

[Step 3/5] Extracting metrics from PDF...

  Company : TESLA
  PDF     : tesla 22 10-k.pdf
  Year    : 2022  |  Columns: [2022, 2021, 2020]
  Pages   : [48, 49, 80]

[Step 1/4] Extracting financial lines from PDF...
  Opened PDF: tesla 22 10-k.pdf (251 pages total)
  Extracted 66 candidate financial lines (74 rows skipped)

[Step 2/4] Assigning fiscal years to numeric columns...
  Year mapping complete: 66 rows with 3-year values

[Step 3/4] Saving raw extraction CSV for inspection...
  [Debug CSV saved] -> ..\data\Tesla\tesla_2022_raw_extraction.csv

[Step 4/4] Normalizing labels using mapping config...

Normalizing 66 rows for 'tesla'...
  [exact]  'cash and cash equiv

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5740.58it/s]


  Found existing collection 'tesla_2022' (2103 documents)
  Collection already populated. Skipping embedding step.
  (Pass rebuild=True to force re-embedding)

  Pipeline complete: TESLA FY2022
  Metrics ready    : ['cash_and_equivalents', 'total_assets', 'total_liabilities', 'total_revenue', 'gross_profit', 'research_and_development', 'selling_general_administrative', 'total_operating_expenses', 'operating_income', 'net_income']
  Vector store     : 2103 documents in ChromaDB



In [4]:
tesla_score = run_evaluation(
    eval_path       = "../data/eval/tesla_eval.json",
    pipeline_result = tesla_result,
    company         = "Tesla",
)



EVALUATION REPORT  TESLA

[PASS] What was Tesla's total revenue in 2022?
       Expected: 81462.0 | Found in response: [2022.0, 81462.0, 2022.0, 81462.0, 2022.0, 1.0, 2022.0, 81462.0, 1.0, 49.0]

[PASS] What was Tesla's gross margin in 2022?
       Expected: 25.6 | Found in response: [2022.0, 25.6, 2022.0, 20853.0, 2022.0, 81462.0, 100.0, 20853.0, 81462.0, 100.0, 25.6, 2022.0, 25.6, 25.6, 1.0, 49.0, 2022.0]

[PASS] What was Tesla's net income in 2022?
       Expected: 12587.0 | Found in response: [2022.0, 12587.0, 2022.0, 12587.0, 12587.0, 2022.0, 15.45, 12587.0, 81462.0, 100.0, 1.0, 50.0]

[PASS] What was Tesla's operating margin in 2022?
       Expected: 16.76 | Found in response: [2022.0, 16.76, 2022.0, 13656.0, 2022.0, 81462.0, 100.0, 13656.0, 81462.0, 100.0, 16.76, 2022.0, 16.76, 1.0, 49.0, 3.0, 2.0, 10.0, 31.0, 2022.0]

[PASS] What was Tesla's net margin in 2022?
       Expected: 15.45 | Found in response: [2022.0, 15.45, 2022.0, 12587.0, 2022.0, 81462.0, 100.0, 12587.0, 81462.0

In [9]:
print("\nOVERALL SYSTEM ACCURACY")
print(f"  Apple: {apple_score['passed']}/{apple_score['total']}  ({apple_score['accuracy']:.1f}%)")
print(f"  Tesla: {tesla_score['passed']}/{tesla_score['total']}  ({tesla_score['accuracy']:.1f}%)")

total_passed = apple_score['passed'] + tesla_score['passed']
total_qs     = apple_score['total']  + tesla_score['total']
print(f"  Combined: {total_passed}/{total_qs}  ({total_passed/total_qs*100:.1f}%)")



OVERALL SYSTEM ACCURACY
  Apple: 6/6  (100.0%)
  Tesla: 6/6  (100.0%)
  Combined: 12/12  (100.0%)


now we are doing for GraphRAG , entiity and relationship extraction and testing them

In [2]:
import neo4j
from neo4j import GraphDatabase

driver = GraphDatabase.driver(
    "bolt://localhost:7687",
    auth=("neo4j", "sudhendra@123")
)

with driver.session() as session:
    result = session.run("RETURN 'connected' AS status")
    print(result.single()["status"])

driver.close()


connected


In [4]:
import sys
sys.path.insert(0, '..')

from src.graph_store import GraphStore

gs = GraphStore("bolt://127.0.0.1:7687", "neo4j", "sudhendra@123")

gs.add_entity("Apple", "Company", "apple")
gs.add_entity("TSMC", "Company", "apple")
gs.add_entity("Taiwan", "Location", "apple")
gs.add_entity("Supply Chain Concentration", "Risk", "apple")

gs.add_relationship("Apple", "RELIES_ON", "TSMC", "apple")
gs.add_relationship("TSMC", "LOCATED_IN", "Taiwan", "apple")
gs.add_relationship("Apple", "FACES_RISK", "Supply Chain Concentration", "apple")

context = gs.get_entity_context("Apple", "apple")
for row in context:
    print(row)

gs.close()


{'source': 'Apple', 'path': ['RELIES_ON'], 'target': 'TSMC', 'target_type': 'Company'}
{'source': 'Apple', 'path': ['RELIES_ON', 'LOCATED_IN'], 'target': 'Taiwan', 'target_type': 'Location'}
{'source': 'Apple', 'path': ['FACES_RISK'], 'target': 'Supply Chain Concentration', 'target_type': 'Risk'}


In [1]:
import sys
sys.path.insert(0, '..')

from src.graph_extractor import extract_graph_elements

sample_chunk = """
Apple relies on third-party manufacturers, primarily in Asia, including Foxconn and TSMC, 
to manufacture its products. The Company competes with Samsung, Google, and Microsoft 
in key markets. Tim Cook serves as Chief Executive Officer. 
Apple faces significant risks from supply chain concentration and geopolitical tensions 
related to Taiwan and China.
"""

result = extract_graph_elements(sample_chunk)

print("ENTITIES:")
for e in result["entities"]:
    print(f"  {e['name']} ({e['type']})")

print("\nRELATIONSHIPS:")
for r in result["relationships"]:
    print(f"  {r['source']} --{r['type']}--> {r['target']}")


ENTITIES:
  Apple (Company)
  Foxconn (Company)
  TSMC (Company)
  Samsung (Company)
  Google (Company)
  Microsoft (Company)
  Taiwan (Location)
  China (Location)
  Tim Cook (Person)
  Supply Chain Concentration (Risk)

RELATIONSHIPS:
  Apple --RELIES_ON--> TSMC
  Apple --RELIES_ON--> Foxconn
  Apple --COMPETES_WITH--> Samsung
  Apple --COMPETES_WITH--> Google
  Apple --COMPETES_WITH--> Microsoft
  Tim Cook --LEADS--> Apple
  Apple --FACES_RISK--> Supply Chain Concentration
  Apple --LOCATED_IN--> Taiwan
  Apple --OPERATES_IN--> China


In [2]:
import sys
sys.path.insert(0, '..')
from pipeline import run_full_pipeline
from src.hybrid_ask import hybrid_ask

apple_result = run_full_pipeline("apple", "../data/Apple/Apple23 report.pdf", 2023, rebuild_vectors=False)

print(hybrid_ask("Who is the CEO of Apple?", "apple", apple_result, "bolt://127.0.0.1:7687", "neo4j", "sudhendra@123"))


d:\sudhendra\learning projects\GraphRAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



  FULL PIPELINE: APPLE  |  FY2023

[Step 1/5] Detecting financial statement pages...
  Auto-detected income statement pages : [30, 31, 49, 57]
  Auto-detected balance sheet pages    : [33, 52]
  Combined target pages: [30, 31, 33, 49, 52, 57]

[Step 2/5] Loading/generating metric mapping config...
  Existing mapping config found: apple.json

[Step 3/5] Extracting metrics from PDF...

  Company : APPLE
  PDF     : Apple23 report.pdf
  Year    : 2023  |  Columns: [2023, 2022, 2021]
  Pages   : [30, 31, 33, 49, 52, 57]

[Step 1/4] Extracting financial lines from PDF...
  Opened PDF: Apple23 report.pdf (80 pages total)
  Extracted 95 candidate financial lines (139 rows skipped)

[Step 2/4] Assigning fiscal years to numeric columns...
  Year mapping complete: 95 rows with 3-year values

[Step 3/4] Saving raw extraction CSV for inspection...
  [Debug CSV saved] -> ..\data\Apple\apple_2023_raw_extraction.csv

[Step 4/4] Normalizing labels using mapping config...

Normalizing 95 rows for 'app

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9207.84it/s]


  Found existing collection 'apple_2023' (666 documents)
  Collection already populated. Skipping embedding step.
  (Pass rebuild=True to force re-embedding)

  Pipeline complete: APPLE FY2023
  Metrics ready    : ['total_revenue', 'gross_profit', 'research_and_development', 'selling_general_administrative', 'total_operating_expenses', 'operating_income', 'net_income', 'cash_and_equivalents', 'total_assets', 'total_liabilities']
  Vector store     : 666 documents in ChromaDB

 The CEO of Apple is Timothy D. Cook [IS_CEO_OF] Apple Inc., according to the provided information sources. This information comes from the LEADERSHIP & PEOPLE category.


In [3]:
import importlib, sys
for mod in ['src.graph_store', 'src.graph_ask', 'src.hybrid_ask']:
    if mod in sys.modules:
        importlib.reload(sys.modules[mod])

from src.hybrid_ask import hybrid_ask

print(hybrid_ask("Who is the CEO of Apple?", "apple", apple_result,
                 "bolt://127.0.0.1:7687", "neo4j", "sudhendra@123"))


 The CEO of Apple is Luca Maestri. This information can be found in the Document Text under the signed statement by Luca Maestri as Senior Vice President and Chief Financial Officer of Apple Inc. (Deterministic Calculation).


In [4]:
import importlib, sys
for mod in ['src.graph_store', 'src.graph_ask', 'src.hybrid_ask']:
    if mod in sys.modules:
        importlib.reload(sys.modules[mod])

from src.hybrid_ask import hybrid_ask

print(hybrid_ask("Who is the CEO of Apple?", "apple", apple_result,
                 "bolt://127.0.0.1:7687", "neo4j", "sudhendra@123"))


 The CEO of Apple is Timothy D. Cook [IS_CEO_OF] Apple Inc., according to the provided information sources. This information comes from the LEADERSHIP & PEOPLE category.


In [5]:
import sys
sys.path.insert(0, '..')

from pipeline import run_full_pipeline
from src.hybrid_ask import hybrid_ask

microsoft_result = run_full_pipeline(
    company_name    = "microsoft",
    pdf_path        = "../data/Microsoft/microsoft_23'-10k.pdf",
    year            = 2023,
    rebuild_vectors = False,
)

NEO4J = {"neo4j_uri": "bolt://127.0.0.1:7687", "neo4j_user": "neo4j", "neo4j_pass": "sudhendra@123"}

questions = [
    "What are the main risks Microsoft faces?",
    "Who is the CEO of Microsoft?",
    "What products does Microsoft sell?",
    "What is Microsoft's exposure to China?",
]

for q in questions:
    print(f"\nQ: {q}")
    print("-" * 60)
    print(hybrid_ask(q, "microsoft", microsoft_result, year=2023, **NEO4J))
    print()



  FULL PIPELINE: MICROSOFT  |  FY2023

[Step 1/5] Detecting financial statement pages...
  Auto-detected income statement pages : [58, 59]
  Auto-detected balance sheet pages    : [60, 87, 96]
  Combined target pages: [58, 59, 60, 87, 96]

[Step 2/5] Loading/generating metric mapping config...
  Existing mapping config found: microsoft.json

[Step 3/5] Extracting metrics from PDF...

  Company : MICROSOFT
  PDF     : microsoft_23'-10k.pdf
  Year    : 2023  |  Columns: [2023, 2022, 2021]
  Pages   : [58, 59, 60, 87, 96]

[Step 1/4] Extracting financial lines from PDF...
  Opened PDF: microsoft_23'-10k.pdf (116 pages total)
  Extracted 81 candidate financial lines (89 rows skipped)

[Step 2/4] Assigning fiscal years to numeric columns...
  Year mapping complete: 81 rows with 3-year values

[Step 3/4] Saving raw extraction CSV for inspection...
  [Debug CSV saved] -> ..\data\Microsoft\microsoft_2023_raw_extraction.csv

[Step 4/4] Normalizing labels using mapping config...

Normalizing 81

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6110.77it/s]


  Found existing collection 'microsoft_2023' (904 documents)
  Collection already populated. Skipping embedding step.
  (Pass rebuild=True to force re-embedding)

  Pipeline complete: MICROSOFT FY2023
  Metrics ready    : ['total_revenue', 'gross_profit', 'research_and_development', 'selling_general_administrative', 'operating_income', 'net_income', 'cash_and_equivalents', 'total_assets', 'total_liabilities']
  Vector store     : 904 documents in ChromaDB


Q: What are the main risks Microsoft faces?
------------------------------------------------------------
 Based on the provided information, Microsoft faces several risks that can adversely affect its business, financial condition, results of operations, cash flows, and the trading price of its common stock.

1. Strategic and Competitive Risks (Knowledge Graph):
   - Intense competition across all markets for its products and services may lead to lower revenue or operating margins. This risk is related to the competitive nature of t

comparison check

In [6]:
from src.compare_ask import compare_ask

result = compare_ask(
    question   = "What are the main risks this company faces?",
    company_a  = "apple",  result_a = apple_result,  year_a = 2023,
    company_b  = "microsoft", result_b = microsoft_result, year_b = 2023,
    neo4j_uri  = "bolt://127.0.0.1:7687",
    neo4j_user = "neo4j",
    neo4j_pass = "sudhendra@123",
)
print(result)


Analyzing apple...
Analyzing microsoft...
 1. APPLE Summary: Apple faces significant risks from industrial accidents, public health issues, political uncertainty, natural disasters, international challenges, investment transactions, data breaches, and supply chain disruptions. These risks could impact its business operations, financial condition, and reputation.

2. MICROSOFT Summary: Microsoft operates in a highly competitive market and faces strategic and competitive risks, risks related to the evolution of its business (including investments in research, development, and marketing), legal and regulatory risks, litigation risks, and risks associated with the evolution of technology. These risks could affect its financial condition, reputation, and results of operations.

3. Key Differences: Apple's main risks seem to be more diversified, including supply chain issues, natural disasters, and geopolitical risks that are not as prominent in Microsoft's risk profile. On the other hand, M